# Column-Mask Performance Diagnostic

**Goal.** Find the **tables and mask functions that actually hurt query performance** — not just what
is masked. It attributes real query time (from `system.query.history` + lineage) to tables and to the
mask functions applied on them, flags the expensive kind (Python / non-deterministic), surfaces the
slowest queries with their bottleneck signals, and reads a query plan to prove whether masking is in
the critical path.

**100% read-only.** Every cell is a `SELECT`/`EXPLAIN` against `system.*` and `information_schema`.
It **creates nothing** and is safe in production.

**Requires** the `system.query`, `system.access` (lineage), and `system.information_schema` system
schemas to be enabled (standard on Unity Catalog).

**Parameters (edit inline):** each cell uses `INTERVAL 3 DAYS` for the look-back and
`execution_duration_ms > 2000` (2s) as the "slow" threshold. Widen/narrow to taste. To focus on one
catalog, add `WHERE source_table_full_name LIKE 'your_catalog.%'`.

**Tip:** for a large estate, run this on a **SQL warehouse** (Sections 1–2 join `query.history` to
`table_lineage`, which is heavy on a small serverless notebook). Start at 3 days, then widen once it's
responsive.

## 1. Tables that hurt the most
Ranks tables by the total query time spent reading them (slow `SELECT`s, last 7 days), and flags which
are masked. This is the "where is the time going" view. `is_masked = true` **and** high `total_exec_s`
is the only combination where column masking could plausibly be implicated — confirm it in Sections 3 & 5.

In [0]:
WITH slow AS (
  SELECT statement_id, execution_duration_ms, read_bytes
  FROM system.query.history
  WHERE start_time > current_timestamp() - INTERVAL 3 DAYS
    AND statement_type = 'SELECT' AND execution_status = 'FINISHED'
    AND from_result_cache = false AND execution_duration_ms > 2000
),
lin AS (
  SELECT DISTINCT statement_id, source_table_full_name
  FROM system.access.table_lineage
  WHERE event_date > current_date() - INTERVAL 4 DAYS AND source_table_full_name IS NOT NULL
),
masked AS (
  SELECT DISTINCT table_catalog||'.'||table_schema||'.'||table_name AS tbl
  FROM system.information_schema.column_masks
)
SELECT
  l.source_table_full_name                       AS table_name,
  (l.source_table_full_name IN (SELECT tbl FROM masked)) AS is_masked,
  count(*)                                        AS slow_queries,
  round(sum(s.execution_duration_ms)/1000.0, 1)   AS total_exec_s,
  round(avg(s.execution_duration_ms)/1000.0, 1)   AS avg_exec_s,
  round(sum(s.read_bytes)/1e9, 1)                 AS total_read_gb
FROM slow s
JOIN lin l USING (statement_id)
GROUP BY 1, 2
ORDER BY total_exec_s DESC
LIMIT 30;

## 2. Mask functions that hurt the most
Attributes that same slow-query time to the **mask function** applied on each hot table, and shows the
function's language. This answers "which masking functions sit on the heaviest workloads." (It measures
*exposure* — a function on a hot table — not proof the function itself is the cost; Sections 3 & 5 test that.)

In [0]:
WITH slow AS (
  SELECT statement_id, execution_duration_ms
  FROM system.query.history
  WHERE start_time > current_timestamp() - INTERVAL 3 DAYS
    AND statement_type = 'SELECT' AND execution_status = 'FINISHED'
    AND from_result_cache = false AND execution_duration_ms > 2000
),
lin AS (
  SELECT DISTINCT statement_id, source_table_full_name
  FROM system.access.table_lineage
  WHERE event_date > current_date() - INTERVAL 4 DAYS AND source_table_full_name IS NOT NULL
),
tbl_cost AS (
  SELECT l.source_table_full_name AS tbl, sum(s.execution_duration_ms) AS exec_ms, count(*) AS q
  FROM slow s JOIN lin l USING (statement_id) GROUP BY 1
),
mask AS (
  SELECT table_catalog||'.'||table_schema||'.'||table_name AS tbl, mask_name
  FROM system.information_schema.column_masks
)
SELECT
  m.mask_name                                     AS mask_function,
  coalesce(r.external_language, 'SQL or built-in') AS language,
  r.is_deterministic,
  count(DISTINCT m.tbl)                           AS tables_masked,
  round(sum(c.exec_ms)/1000.0, 1)                 AS hot_table_exec_s,
  sum(c.q)                                        AS slow_queries_on_those_tables
FROM mask m
JOIN tbl_cost c USING (tbl)
LEFT JOIN system.information_schema.routines r
  ON concat(r.routine_catalog, '.', r.routine_schema, '.', r.routine_name) = m.mask_name
GROUP BY 1, 2, 3
ORDER BY hot_table_exec_s DESC
LIMIT 20;

## 3. The expensive kind: Python and/or non-deterministic mask functions
Python UDF masks are ~17× slower than SQL and force queries **off Photon**; non-deterministic functions
block optimizations. **If this returns no rows, none of your masks are the slow kind** — masking is not
your performance problem, and the hot tables in Section 1 are slow for other reasons (see Section 4).

In [0]:
SELECT DISTINCT
  r.routine_catalog, r.routine_schema, r.routine_name,
  r.external_language, r.is_deterministic
FROM system.information_schema.routines r
JOIN (SELECT DISTINCT mask_name AS fn FROM system.information_schema.column_masks) m
  ON concat(r.routine_catalog, '.', r.routine_schema, '.', r.routine_name) = m.fn
WHERE (r.external_language = 'Python' OR lower(r.is_deterministic) IN ('no', 'false'))
  AND r.routine_catalog <> 'system'   -- exclude Databricks' own built-in system masks (not yours to change)
ORDER BY r.external_language DESC NULLS LAST, r.is_deterministic;

## 4. Slowest queries — where is the time actually going?
The bottleneck signals for the heaviest queries. Look for **spill** (memory pressure), **shuffle**
(joins/aggregations), low **cache_pct** (cold reads), and low **pruned_files** (weak
partitioning/clustering). These — not SQL masks — are the usual causes. (`statement_preview` may show
`<REDACTED>` if your own governance masks the query-text column.)

In [0]:
SELECT
  round(execution_duration_ms/1000.0, 1)   AS exec_s,
  round(compilation_duration_ms/1000.0, 1) AS compile_s,
  round(read_bytes/1e9, 2)                 AS read_gb,
  read_rows,
  round(spilled_local_bytes/1e9, 2)        AS spill_gb,
  round(shuffle_read_bytes/1e9, 2)         AS shuffle_gb,
  read_io_cache_percent                    AS cache_pct,
  read_files, pruned_files,
  executed_by,
  left(replace(statement_text, '\n', ' '), 90) AS statement_preview
FROM system.query.history
WHERE start_time > current_timestamp() - INTERVAL 3 DAYS
  AND statement_type = 'SELECT' AND execution_status = 'FINISHED'
  AND from_result_cache = false
ORDER BY execution_duration_ms DESC
LIMIT 25;

## 5. Read the plan — is masking in the critical path?
`EXPLAIN FORMATTED` shows exactly how a mask is applied. **What to look for:**

- **`PhotonSecureView` / "fully supported by Photon"** → a SQL/built-in mask, applied natively and cheaply. Masking is **not** the bottleneck.
- **`BatchEvalPython` / `PythonUDF`** → a **Python** mask. This runs row-by-row, is ~17× slower, and drops the query off Photon. This is a real cost — rewrite as SQL.
- **`missing`/`partial` statistics** in the plan → run `ANALYZE TABLE … COMPUTE STATISTICS` (a common real cause of slowness).

The example below runs on `system.access.audit` (present in every workspace). **Swap in your own top
masked table + column from Section 1** to inspect it.

In [0]:
EXPLAIN FORMATTED
SELECT request_params
FROM system.access.audit
WHERE event_date > current_date() - INTERVAL 1 DAYS;

## 6. Mask density — worst case for `SELECT *`
Even cheap SQL masks add up if a query selects many masked columns. These tables are the worst case for
`SELECT *`; prefer explicit column lists that avoid masked columns you don't need.

In [0]:
WITH total AS (
  SELECT table_catalog, table_schema, table_name, count(*) AS total_columns
  FROM system.information_schema.columns GROUP BY 1, 2, 3
),
masked AS (
  SELECT table_catalog, table_schema, table_name, count(*) AS masked_columns
  FROM system.information_schema.column_masks GROUP BY 1, 2, 3
)
SELECT
  m.table_catalog, m.table_schema, m.table_name,
  m.masked_columns, t.total_columns,
  round(100.0 * m.masked_columns / nullif(t.total_columns, 0), 1) AS pct_masked
FROM masked m LEFT JOIN total t USING (table_catalog, table_schema, table_name)
ORDER BY m.masked_columns DESC
LIMIT 30;

## 7. Maintenance & Predictive Optimization — are OPTIMIZE / VACUUM / ANALYZE running?
Stale or missing table maintenance is one of the most common causes of slow queries: no `OPTIMIZE`/liquid
clustering means small files and poor pruning, and stale `ANALYZE` means the optimizer plans blind.
[Predictive Optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization) runs
these for you on managed tables. **This first query shows whether it is running at all** (last 30 days).
If it returns no rows, no automatic maintenance is happening anywhere.

In [0]:
SELECT
  operation_type,
  count(*)                                                            AS operations,
  count(DISTINCT concat(catalog_name,'.',schema_name,'.',table_name)) AS tables,
  date(max(end_time))                                                 AS most_recent
FROM system.storage.predictive_optimization_operations_history
WHERE end_time > current_timestamp() - INTERVAL 30 DAYS
GROUP BY operation_type
ORDER BY operations DESC;

### Are your hottest tables actually being maintained?
Joins your hot tables (as in Section 1) to their last `OPTIMIZE`/`VACUUM`/`ANALYZE`. A blank date or
**`NOT maintained by PO`** on a hot table — or a `last_analyze` months behind `last_optimize` — is a
direct "you are not doing what you need" signal. (Databricks-managed `system.*` tables are expected to
show as not-PO-maintained; focus on your own catalogs. Manually-run maintenance appears in Section 7c.)

In [0]:
WITH slow AS (
  SELECT statement_id, execution_duration_ms FROM system.query.history
  WHERE start_time > current_timestamp() - INTERVAL 3 DAYS AND statement_type = 'SELECT'
    AND execution_status = 'FINISHED' AND from_result_cache = false AND execution_duration_ms > 2000
),
lin AS (
  SELECT DISTINCT statement_id, source_table_full_name FROM system.access.table_lineage
  WHERE event_date > current_date() - INTERVAL 4 DAYS AND source_table_full_name IS NOT NULL
),
hot AS (
  SELECT l.source_table_full_name AS tbl, count(*) AS slow_queries,
         round(sum(s.execution_duration_ms)/1000.0, 1) AS total_exec_s
  FROM slow s JOIN lin l USING (statement_id) GROUP BY 1
),
maint AS (
  SELECT lower(concat(catalog_name,'.',schema_name,'.',table_name)) AS tbl,
         max(CASE WHEN operation_type IN ('COMPACTION','CLUSTERING') THEN end_time END) AS last_optimize,
         max(CASE WHEN operation_type = 'VACUUM' THEN end_time END)                     AS last_vacuum,
         max(CASE WHEN operation_type = 'ANALYZE' THEN end_time END)                    AS last_analyze
  FROM system.storage.predictive_optimization_operations_history
  WHERE operation_status = 'SUCCESSFUL' GROUP BY 1
)
SELECT h.tbl, h.slow_queries, h.total_exec_s,
       date(m.last_optimize) AS last_optimize,
       date(m.last_vacuum)   AS last_vacuum,
       date(m.last_analyze)  AS last_analyze,
       CASE WHEN m.tbl IS NULL THEN 'NOT maintained by PO' ELSE 'maintained' END AS maintenance
FROM hot h LEFT JOIN maint m ON lower(h.tbl) = m.tbl
ORDER BY h.total_exec_s DESC
LIMIT 30;

### 7c. Manual maintenance (if you don't use Predictive Optimization)
If Section 7's PO history is sparse, check whether maintenance is being run by hand. Nothing here **and**
nothing in the PO history means the table is not being maintained at all.

In [0]:
SELECT
  regexp_extract(statement_text, '(?i)^\\s*(optimize|vacuum|analyze)', 1) AS operation,
  count(*)              AS runs,
  date(max(start_time)) AS most_recent
FROM system.query.history
WHERE (statement_text ILIKE 'OPTIMIZE %' OR statement_text ILIKE 'VACUUM %' OR statement_text ILIKE 'ANALYZE %')
  AND start_time > current_timestamp() - INTERVAL 30 DAYS
GROUP BY 1
ORDER BY runs DESC;

## How to read your results

1. **Section 3 returns no rows** (no Python / non-deterministic masks) → column masking is **not** your
   bottleneck. Your masks are SQL/built-in, applied as Photon-native secure views (Section 5). Focus the
   investigation on the hot tables in Section 1 using the Section 4 signals: **spill** → bigger warehouse
   / less shuffle; **cold cache / low pruning** → `OPTIMIZE`, liquid clustering, better partitioning,
   `ANALYZE … COMPUTE STATISTICS`; heavy **shuffle** → join/aggregation tuning.
2. **Section 3 returns Python or non-deterministic masks** → cross-check them against the hot tables in
   Sections 1–2. Where a Python mask sits on a hot table, that is a genuine cost — rewrite it as a
   deterministic SQL function:
   ```sql
   CREATE OR REPLACE FUNCTION mask_string(v STRING)
   RETURNS STRING
   RETURN CASE WHEN is_account_group_member('admins') THEN v ELSE '***' END;
   ```
3. **High-density tables (Section 6)** in heavy `SELECT *` workloads → switch to explicit column lists.